# Utilização da extensão 'Parquet' - Transformando arquivos 'csv' em 'parquet': Criado pelo Nicholas Birochi

#### Importando bibliotecas essenciais para a Extração de Dados:

In [ ]:
# Importando bibliotecas:
from glob import glob  # Busca de arquivos por padrões...
import pandas as pd    # Análise e manipulação de dados...
import os              # Operações com arquivos e sistema...

#### Criando as funções necessárias para esse processo:

In [ ]:
# Função para a conversão de cada uma das colunas em formato "string", assim não perdendo nenhum dado:
def converter_colunas_mistas_para_str(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte colunas 'object' que tenham tipos mistos (ex.: int + str) para string,
    preservando NaN como NaN (não vira "nan").
    Obs.: Se você já lê com dtype=str, isso tende a ser redundante, mas mantém a proteção.
    """
    for col in df.columns:
        if df[col].dtype != "object":
            continue

        s = df[col]
        # checagem rápida por amostra (barata)
        amostra = s.dropna().head(200)
        if amostra.empty:
            continue

        tipos = amostra.map(type).nunique()
        if tipos > 1:
            mask = s.notna()
            df.loc[mask, col] = s.loc[mask].astype(str)

    return df


def ler_sem_colunas(path: str, colunas_excluir, **kwargs) -> pd.DataFrame:
    """
    Lê CSV ignorando colunas por índice (int) OU por nome (str).
    Faz um header-read (nrows=0) para descobrir as colunas disponíveis.
    """
    colunas = pd.read_csv(path, nrows=0, **kwargs).columns.tolist()

    if all(isinstance(c, int) for c in colunas_excluir):
        colunas_ler = [col for i, col in enumerate(colunas) if i not in colunas_excluir]
    else:
        colunas_ler = [col for col in colunas if col not in colunas_excluir]

    return pd.read_csv(path, usecols=colunas_ler, **kwargs)


# ========= 2) Otimização: object -> category (safe) =========

def otimizar_obj_para_category_safe(
    df: pd.DataFrame,
    limite_unicos: float = 0.2,
    dropna: bool = False,
    somente_strings: bool = True,
    amostra_max: int = 200_000,
) -> pd.DataFrame:
    """
    Converte colunas object para category quando fizer sentido (baixa cardinalidade),
    com proteção para não transformar colunas "mistas" nem estourar memória.
    """
    obj_cols = df.dtypes[df.dtypes == "object"].index.tolist()
    n = len(df)
    if n == 0 or not obj_cols:
        return df

    k = min(amostra_max, n)
    step = max(1, n // k)

    for c in obj_cols:
        s = df[c]

        # evita object mista (ex.: números + strings)
        if somente_strings:
            amostra_tipo = s.dropna().head(50)
            if len(amostra_tipo) and not amostra_tipo.map(lambda x: isinstance(x, str)).all():
                continue

        # 1) estimativa rápida por amostra
        amostra = s.iloc[::step]
        try:
            ratio_est = amostra.nunique(dropna=dropna) / max(1, len(amostra))
        except MemoryError:
            continue

        if ratio_est >= limite_unicos:
            continue

        # 2) confirma no full
        try:
            ratio_full = s.nunique(dropna=dropna) / n
        except MemoryError:
            continue

        if ratio_full < limite_unicos:
            try:
                df[c] = s.astype("category")
            except MemoryError:
                pass

    return df


# ========= 3) Pipeline: CSV -> Parquet (com otimização antes de salvar) =========

def _preparar_df_para_parquet(
    df: pd.DataFrame,
    limite_unicos: float = 0.2,
    dropna: bool = False,
    somente_strings: bool = True,
    amostra_max: int = 200_000,
) -> pd.DataFrame:
    # Correção específica (mantida da sua lógica)
    if "Movimento de Colaboradores" in df.columns:
        mask = df["Movimento de Colaboradores"].notna()
        df.loc[mask, "Movimento de Colaboradores"] = df.loc[mask, "Movimento de Colaboradores"].astype(str)

    # Proteção para colunas com tipos mistos
    df = converter_colunas_mistas_para_str(df)

    # Otimização de dtypes (reduz tamanho/memória e costuma melhorar parquet)
    df = otimizar_obj_para_category_safe(
        df,
        limite_unicos=limite_unicos,
        dropna=dropna,
        somente_strings=somente_strings,
        amostra_max=amostra_max,
    )
    return df


def converter_csv_para_parquet(
    pasta_csv: str,
    tipo: str,
    colunas_excluir=None,
    encoding: str = "latin1",
    compression: str = "zstd",              # zstd costuma comprimir melhor; snappy = mais rápido
    engine: str = "pyarrow",
    limite_unicos: float = 0.2,
    dropna: bool = False,
    somente_strings: bool = True,
    amostra_max: int = 200_000,
) -> None:
    """
    Converte todos os CSVs de uma pasta para Parquet, aplicando:
    - leitura robusta (dtype=str)
    - correções e proteção de tipos
    - otimização object -> category (quando baixa cardinalidade)
    - escrita parquet com compressão
    """
    # Define o caminho de saída dependendo do tipo
    tipo = tipo.upper().strip()
    if tipo == "E":
        pasta_parquet = r"..\..\Base de Dados Original\parquet\Executivos"
    elif tipo == "M":
        pasta_parquet = r"..\..\Base de Dados Original\parquet\Mensalistas"
    elif tipo == "H":
        pasta_parquet = r"..\..\Base de Dados Original\parquet\Horistas"
    else:
        raise ValueError("Tipo deve ser 'E' ou 'M' ou 'H'")

    os.makedirs(pasta_parquet, exist_ok=True)

    arquivos_csv = glob(os.path.join(pasta_csv, "*.csv"))
    if not arquivos_csv:
        print(f"⚠️ Nenhum CSV encontrado em: {pasta_csv}")
        return

    for caminho in arquivos_csv:
        try:
            # Leitura (mantendo sua lógica de excluir colunas)
            if colunas_excluir:
                df = ler_sem_colunas(
                    caminho,
                    colunas_excluir,
                    encoding=encoding,
                    dtype=str,
                    low_memory=False,
                )
            else:
                df = pd.read_csv(
                    caminho,
                    encoding=encoding,
                    dtype=str,
                    low_memory=False,
                )

            # Prepara + otimiza ANTES de salvar
            df = _preparar_df_para_parquet(
                df,
                limite_unicos=limite_unicos,
                dropna=dropna,
                somente_strings=somente_strings,
                amostra_max=amostra_max,
            )

            # Nome e caminho de saída
            nome_base = os.path.splitext(os.path.basename(caminho))[0]
            caminho_salvar = os.path.join(pasta_parquet, f"{nome_base}.parquet")

            # Escrita parquet otimizada
            df.to_parquet(
                caminho_salvar,
                index=False,
                engine=engine,
                compression=compression,
            )

            print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"⚠️ Erro ao processar {caminho}: {e}")


# ========= 4) Exportador de parquets (dict de DFs) com a mesma otimização =========

def exportar_parquets(
    dfs: dict[str, pd.DataFrame],
    base_dir: str = r"..\project\database",
    compression: str = "zstd",
    engine: str = "pyarrow",
    limite_unicos: float = 0.2,
    dropna: bool = False,
    somente_strings: bool = True,
    amostra_max: int = 200_000,
) -> None:
    os.makedirs(base_dir, exist_ok=True)

    for filename, df in dfs.items():
        try:
            df = _preparar_df_para_parquet(
                df,
                limite_unicos=limite_unicos,
                dropna=dropna,
                somente_strings=somente_strings,
                amostra_max=amostra_max,
            )

            path = os.path.join(base_dir, filename)
            if not path.lower().endswith(".parquet"):
                path += ".parquet"

            df.to_parquet(
                path,
                index=False,
                engine=engine,
                compression=compression,
            )
            print(f"✅ Carregado: '{path}'")

        except Exception as e:
            print(f"⚠️ Erro ao salvar '{filename}': {e}")



#### Inserindo os Endereços e nomes para dentro das funções:

In [ ]:
# Exemplo de uso para Executivos:
converter_csv_para_parquet(
    pasta_csv=R'..\..\Base de Dados Original\csv\Executivos',
    tipo='E',
    colunas_excluir=[0, 1, 3, 8, 14, 17],
    compression="zstd",
    limite_unicos=0.15,       # mais agressivo: converte mais colunas para category
)


In [ ]:
# Exemplo de uso para Mensalistas:
converter_csv_para_parquet(
    pasta_csv=R'..\..\Base de Dados Original\csv\Mensalistas',
    tipo='M',
    colunas_excluir=[0, 1, 3, 8, 14, 17],
    compression="zstd",
    limite_unicos=0.15,       # mais agressivo: converte mais colunas para category
)


In [ ]:
# Exemplo de uso para Horistas:
converter_csv_para_parquet(
    pasta_csv=R'..\..\Base de Dados Original\csv\Horistas',
    tipo='H',
    colunas_excluir=[0, 1, 3, 8, 14, 17, 18],
    compression="zstd",
    limite_unicos=0.15,       # mais agressivo: converte mais colunas para category
)
